## Imports

In [15]:
import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import HTML
import joblib
from sklearn.preprocessing import MinMaxScaler

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

## Normalize target 0-4 

In [ ]:
def load_video_score(score_path: str = None, lower_bound: float = 0, upper_bound: float = 4, columns: list[str] = ["score"]) -> pd.DataFrame:
    """
    Loads and scales the video score found in the target file into the range [lower_bound, upper_bound]

    Args:
        score_path: path the the csv-file containing the score column
        lower_bound: The lower bound of the output range
        upper_bound: The upper bound of the output range
        columns: The columns to scale

    Returns:
        DataFrame: A pandas DataFrame with the scaled columns fit to the input range using MinMaxScaler from sklearn as the last column.
    """
    if score_path is None:
        raise Exception("Path cannot be of type 'None'")

    df = pd.read_csv(score_path)

    bad_videos = [
    "A1",
    "B2",
    "B3",
    "B4",
    "B5"
]

    df = df.loc[~df["file"].isin(bad_videos)
                              
]
    for column in columns:
        if column in df.columns:
            if column == "score":
                # Remove faulty scores
                df = df.loc[df["score"] != 0.0]

            scaler = MinMaxScaler(feature_range=(lower_bound, upper_bound))
            df[f"scaled_{column}"] = scaler.fit_transform(df[[column]])
        return df
    else:
        raise Exception(f"File has no column named '{column}'")
    


def select_equally_spaced_frames(data, num_frames=30, axis=0):
    """
    Select equally spaced frames from n-dimensional data along specified axis.
    
    Parameters:
    -----------
    data : numpy array or list
        Input data (2D, 4D, or any dimension)
    num_frames : int
        Number of frames to select (default: 30)
    axis : int
        Axis along which to select frames (default: 0, typically the time/frame dimension)
    
    Returns:
    --------
    numpy array
        Selected frames with same dimensions as input except the selected axis
    """
    # Convert to numpy array if it's a list
    if not isinstance(data, np.ndarray):
        data = np.array(data)
    
    # Get total number of frames along the specified axis
    total_frames = data.shape[axis]
    
    # Generate equally spaced indices
    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    
    # Select using advanced indexing
    # Build a tuple of slices for indexing
    selector = [slice(None)] * data.ndim
    selector[axis] = indices
    
    return data[tuple(selector)]





In [26]:
scaled_df = load_video_score(
    score_path="../../MainProject/data/video_scores.csv",
    lower_bound=0,
    upper_bound=4,
    columns=["score"]
)

scaled_df.to_csv(
    "../../MainProject/data/video_scores_scaled_0_4.csv",
    index=False
)

print("Saved: scores_scaled_0_4.csv")

Saved: scores_scaled_0_4.csv


## Functions

In [33]:
def create_fixed_c_sequence_from_running_column(
    input_folder,
    output_folder,
    score_csv_path,
    C=30
):
    os.makedirs(output_folder, exist_ok=True)

    score_df = pd.read_csv(score_csv_path)

    score_dict = {}

    for _, row in score_df.iterrows():
        video_id = str(row["file"]).split("_")[0]
        score_dict[video_id] = row["scaled_score"]

    joints = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    columns = []

    for frame_idx in range(C):
        for joint in joints:
            columns += [
                f"frame{frame_idx}_{joint}_x",
                f"frame{frame_idx}_{joint}_y",
                f"frame{frame_idx}_{joint}_z"
            ]

    columns.append("target")

    saved_count = 0

    for file_name in os.listdir(input_folder):

        if not file_name.endswith(".csv"):
            continue

        print(f"\nProcessing: {file_name}")

        video_id = file_name.split("_")[0]

        if video_id not in score_dict:
            print(f"Skipping {file_name}: no target found")
            continue

        csv_path = os.path.join(input_folder, file_name)

        df = pd.read_csv(csv_path)

        # -----------------------------------------
        # Trim using running_video column
        # -----------------------------------------

        running_df = df[df["running_video"] == 1]

        if len(running_df) == 0:
            print(f"Skipping {file_name}: no running frames")
            continue

        # -----------------------------------------
        # Remove non-feature columns
        # -----------------------------------------

        drop_cols = ["FrameNo", "running_video"]

        feature_df = running_df.drop(
            columns=[c for c in drop_cols if c in running_df.columns]
        )

        X_trimmed = feature_df.values.astype(np.float32)

        # -----------------------------------------
        # Convert to fixed C frames
        # -----------------------------------------

        if len(X_trimmed) < C:
            print(f"Skipping {file_name}: sequence too short")
            continue

        X_fixed = select_equally_spaced_frames(X_trimmed, num_frames=C, axis=0)


        # -----------------------------------------
        # Target
        # -----------------------------------------

        target = score_dict[video_id]

        # -----------------------------------------
        # Flatten + save
        # -----------------------------------------

        row = X_fixed.flatten().tolist()
        row.append(target)

        output_df = pd.DataFrame([row], columns=columns)

        output_path = os.path.join(
            output_folder,
            f"{video_id}_fixed_c.csv"
        )

        output_df.to_csv(output_path, index=False)

        saved_count += 1

        print(f"Trimmed frames: {len(X_trimmed)}")
        print(f"Fixed shape: {X_fixed.shape}")
        print(f"Target: {target}")
        print(f"Saved: {output_path}")

    print("\nDone.")
    print(f"Saved {saved_count} videos.")

## Paths

In [28]:
input_folder = "../../MainProject/Assignment11/data/mediapipe_not_cut_start_stop"
output_folder = "../../MainProject/data/mediapipe_score_fixed_c"

score_csv_path = "../../MainProject/data/video_scores_scaled_0_4.csv"

## Run trought csv

In [34]:
create_fixed_c_sequence_from_running_column(
    input_folder=input_folder,
    output_folder=output_folder,
    score_csv_path=score_csv_path,
    C=30
)


Processing: A75_mediapipe.csv
Trimmed frames: 98
Fixed shape: (30, 39)
Target: 2.0489483697818276
Saved: ../../MainProject/data/mediapipe_score_fixed_c/A75_fixed_c.csv

Processing: A121_mediapipe.csv
Trimmed frames: 219
Fixed shape: (30, 39)
Target: 3.758899272612682
Saved: ../../MainProject/data/mediapipe_score_fixed_c/A121_fixed_c.csv

Processing: A136_mediapipe.csv
Trimmed frames: 71
Fixed shape: (30, 39)
Target: 1.757045774595765
Saved: ../../MainProject/data/mediapipe_score_fixed_c/A136_fixed_c.csv

Processing: A62_mediapipe.csv
Trimmed frames: 116
Fixed shape: (30, 39)
Target: 3.244541799189422
Saved: ../../MainProject/data/mediapipe_score_fixed_c/A62_fixed_c.csv

Processing: A118_mediapipe.csv
Trimmed frames: 130
Fixed shape: (30, 39)
Target: 2.4268376824860862
Saved: ../../MainProject/data/mediapipe_score_fixed_c/A118_fixed_c.csv

Processing: A66_mediapipe.csv
Trimmed frames: 126
Fixed shape: (30, 39)
Target: 3.2325649757097388
Saved: ../../MainProject/data/mediapipe_score_fix

In [21]:
df = pd.read_csv("../../MainProject/data/mediapipe_score_fixed_c_score/A2_score_fixed_c30.csv")

y = df["target"].values

X_flat = df.drop(columns=["target"]).values

max_frames = 30 
n_features = 39

X = X_flat.reshape(-1, max_frames, n_features)

print(X.shape)

print(X)
print(y)

FileNotFoundError: [Errno 2] No such file or directory: '../../MainProject/data/mediapipe_score_fixed_c_score/A2_score_fixed_c30.csv'